# Systematic Base Model Pipelines and Optimization Experiments

In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from src import base_model as utils
from src import mlflow_utils

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Global Pipeline Configuration Settings

In [2]:
RANDOM_STATE = 42
from pathlib import Path

def _find_project_root():
    """Find the project root by looking for pyproject.toml."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find project root (pyproject.toml)")

ROOT_DIR = str(_find_project_root())
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

In [3]:
# Ingest Transformed Dataset Partitions
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [4]:
# Standard Base Model Search Map Space
models_zoo = {
    "logistic_regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=2000),
    "decision_tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "xgboost": XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1)
}

In [5]:
# Define Constant Immutable Core Column Groups
dummyfy_columns = ['Card Type', 'NumOfProducts', 'Geography', 'Gender']
norm_std_columns = ['Balance', 'Point Earned', 'CreditScore', 'Age', 'Tenure', 'Satisfaction Score', 'EstimatedSalary']

## 2. Experiment 1: Primary Baseline Exploration (With Potential Feature Leakage)

In [6]:
EXP_1_NAME = "customer-churn-simple"
mlflow_utils.init_mlflow_experiment(EXP_1_NAME, DB_PATH, ARTIFACTS_DIR)

'2'

In [7]:
# Configure Experiment Schema Arrays
nomod_columns_exp1 = ['HasCrCard', 'IsActiveMember', 'Complain']

preprocessor_exp1 = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), dummyfy_columns),
        ('num', StandardScaler(), norm_std_columns),
        ('pass', 'passthrough', nomod_columns_exp1)
    ],
    remainder='drop'
)

In [8]:
# Run Pipeline Execution Training Loop
utils.train_and_log_models(
    models=models_zoo, preprocessor=preprocessor_exp1,
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns_exp1
)

2026/06/18 17:41:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 17:41:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/18 17:41:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 17:41:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Successfully processed and logged: logistic_regression
Successfully processed and logged: decision_tree


2026/06/18 17:41:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 17:41:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully processed and logged: random_forest


2026/06/18 17:41:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 17:41:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully processed and logged: xgboost


In [9]:
# Review Performance Metrics Results
mlflow_utils.get_experiment_summary(EXP_1_NAME)

Index(['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time',
       'end_time', 'metrics.accuracy', 'metrics.pr_auc', 'metrics.recall',
       'metrics.roc_auc', 'metrics.f1_score', 'metrics.precision',
       'params.gamma', 'params.reg_lambda', 'params.colsample_bylevel',
       'params.enable_categorical', 'params.sampling_method',
       'params.max_depth', 'params.random_state', 'params.device',
       'params.max_bin', 'params.max_cat_threshold',
       'params.interaction_constraints', 'params.callbacks',
       'params.multi_strategy', 'params.max_cat_to_onehot',
       'params.colsample_bytree', 'params.min_child_weight',
       'params.tree_method', 'params.max_leaves', 'params.scale_pos_weight',
       'params.missing', 'params.colsample_bynode', 'params.feature_types',
       'params.early_stopping_rounds', 'params.num_parallel_tree',
       'params.max_delta_step', 'params.importance_type', 'params.booster',
       'params.n_jobs', 'params.grow_policy', 'param

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,start_time
0,9d854746861242a2b147d6097c402b5f,xgboost,0.9980,0.990291,1.0,0.995122,0.999597,2026-06-18 23:41:21.809000+00:00
1,23a0ca33482b4f909f6429b8434ab0d1,random_forest,0.9980,0.990291,1.0,0.995122,0.999770,2026-06-18 23:41:19.642000+00:00
2,bcf661ba14f541e5bdaf4ecc206592f1,decision_tree,0.9975,0.987893,1.0,0.993910,0.998430,2026-06-18 23:41:17.240000+00:00
3,8972a844a21a4504880d49eb19189c99,logistic_regression,0.9980,0.990291,1.0,0.995122,0.999786,2026-06-18 23:41:14.499000+00:00


> **Analysis Insight:** Perfect classification metrics suggest target leakage from the `Complain` column. Retraining the models without this feature is necessary to capture realistic real-world performance patterns.

## 3. Experiment 2: Robust Evaluation (Excluding Target Leakage Feature)

In [10]:
EXP_2_NAME = "customer-churn-simple-no-complain-feature"
mlflow_utils.init_mlflow_experiment(EXP_2_NAME, DB_PATH, ARTIFACTS_DIR)

'3'

In [11]:
# Sanitize Feature Spaces
nomod_columns_exp2 = ['HasCrCard', 'IsActiveMember']

preprocessor_exp2 = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), dummyfy_columns),
        ('num', StandardScaler(), norm_std_columns),
        ('pass', 'passthrough', nomod_columns_exp2)
    ],
    remainder='drop'
)

In [12]:
# Run Sanitized Pipeline Iteration Execution
utils.train_and_log_models(
    models=models_zoo, preprocessor=preprocessor_exp2,
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns_exp2
)

2026/06/18 17:41:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 17:41:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/18 17:41:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 17:41:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Successfully processed and logged: logistic_regression
Successfully processed and logged: decision_tree


2026/06/18 17:41:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 17:41:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully processed and logged: random_forest


2026/06/18 17:41:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 17:41:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully processed and logged: xgboost


In [13]:
# Review Sanitized Performance Results
mlflow_utils.get_experiment_summary(EXP_2_NAME)

Index(['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time',
       'end_time', 'metrics.accuracy', 'metrics.pr_auc', 'metrics.recall',
       'metrics.roc_auc', 'metrics.f1_score', 'metrics.precision',
       'params.gamma', 'params.reg_lambda', 'params.colsample_bylevel',
       'params.enable_categorical', 'params.sampling_method',
       'params.max_depth', 'params.random_state', 'params.device',
       'params.max_bin', 'params.max_cat_threshold',
       'params.interaction_constraints', 'params.callbacks',
       'params.multi_strategy', 'params.max_cat_to_onehot',
       'params.colsample_bytree', 'params.min_child_weight',
       'params.tree_method', 'params.max_leaves', 'params.scale_pos_weight',
       'params.missing', 'params.colsample_bynode', 'params.feature_types',
       'params.early_stopping_rounds', 'params.num_parallel_tree',
       'params.max_delta_step', 'params.importance_type', 'params.booster',
       'params.n_jobs', 'params.grow_policy', 'param

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,start_time
0,b03bd07f50c64c589ba19ff42a231916,xgboost,0.8540,0.682390,0.531863,0.597796,0.849671,2026-06-18 23:41:30.676000+00:00
1,bb1e3125ab2b4cfd97ca1348f0e65a79,random_forest,0.8755,0.795539,0.524510,0.632201,0.875317,2026-06-18 23:41:28.493000+00:00
2,e2e364a33c8649f98a58704a6b025e72,decision_tree,0.7965,0.501080,0.568627,0.532721,0.711763,2026-06-18 23:41:26.405000+00:00
3,5745380ddfbc4ec1a07e6c844abfd22d,logistic_regression,0.8530,0.743590,0.426471,0.542056,0.838354,2026-06-18 23:41:24.223000+00:00
